In [ ]:
import sys
import os
import pickle
import logging
import numba
import matplotlib
import matplotlib.pyplot as plt
logging.getLogger('matplotlib.font_manager').disabled = True
numba_logger = logging.getLogger('numba')
numba_logger.setLevel(logging.WARNING)

matplotlib_logger = logging.getLogger('matplotlib')
matplotlib_logger.setLevel(logging.WARNING)

# Add the src directory to sys.path
src_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(src_path)

from src.IndependentSteps import Pycromanager2NativeDataType, FFF2NativeDataType, Make_Output_Dir_JF, Make_Analysis_Dir_JF, \
                                    ConsolidateImageShapes, TrimZSlices, AutomaticSpotDetection_JF

from src.SequentialSteps import CellSegmentationStepClass_JF, BIGFISH_SpotDetection, SimpleCellposeSegmentaion, IlluminationCorrection

from src.FinalizationSteps import Save_Outputs, Save_Images, Save_Parameters, Save_Masks, return_to_NAS, remove_local_data_but_keep_h5

from src.Parameters import Parameters, Experiment, Settings, ScopeClass, DataContainer

from src.GeneralOutput import OutputClass

from src.Displays import Display

from src.GUI import GUI, StepGUI

from src.Pipeline import Pipeline

In [ ]:
# List of directories to process
initial_data_location=(
        'smFISH_images/Eric_smFISH_images/20220718/DUSP1_conc_sweep_R2_0min_071422' ,   
        'smFISH_images/Eric_smFISH_images/20220718/DUSP1_conc_sweep_R2_1nM_75min_071422' ,    
        'smFISH_images/Eric_smFISH_images/20220718/DUSP1_conc_sweep_R2_10nM_75min_071422' )
# Parameters
Condition = 'DUSP1_timesweep'      # Experimental condition. The options are:  'GR_timesweep' , 'DUSP1_timesweep' 
Replica = 'H'                   # Replica name. 
time_list  = [0, 75, 75]        # Time of image acquisition
DexConc_list = [0, 1, 10]        # Dex concentration

# Creating the list of dictionaries
DUSP1_CS_R2 = [
    {
        "condition": Condition,
        "replica": Replica,
        "time": time,
        "Dex_Conc": DexConc
    }
    for time, DexConc in zip(time_list, DexConc_list)
]
# Output the result
print(DUSP1_CS_R2)

In [ ]:
# Initialize Parameters
scope = ScopeClass()
scope.voxel_size_yx = 160
scope.spot_yx = 300
scope.spot_z = 500 
data = DataContainer() # you can also initialize these with parameters, but it is not necessary due to defaults
settings = Settings(name='ER_Dec0324') # you also must give a name for the analysis your are doing
#settings.independent_params = DUSP1_CS_R2
experiment = Experiment()
experiment.initial_data_location = initial_data_location
experiment.snr_threshold = 4

settings.load_in_mask = False
experiment.FISHChannel = 0
experiment.nucChannel = 2
experiment.cytoChannel = 1
experiment.voxel_size_z = 500

settings.cellpose_min_size = 100
settings.cellpose_diameter = [180, 90] # most of these options can be done for individually cyto and nuc segmentation, 
                                        # and a list can be or a single float can be passed for both
                                        # always in the order cyto, nuc
settings.cellpose_pretrained_model = [r"/Users/ericron/Desktop/FISH_Processing-1/models/GAPDH_cyto", r'/Users/ericron/Desktop/FISH_Processing-1/models/DAPI_nuclei'] 

In [ ]:
# You can check that all the manditory parameters are set by calling validate
Parameters.validate()

In [ ]:
FFF2NativeDataType()
#IlluminationCorrection()
SimpleCellposeSegmentaion()
BIGFISH_SpotDetection()
#Save_Masks()
#Save_Parameters()
#Save_Outputs()

In [ ]:
pipeline = Pipeline()

In [ ]:
pipeline.run()